In [2]:
%pwd

'/workspace'

In [3]:
!git clone https://github.com/fouzul-hassan/gated-e2t/

fatal: destination path 'gated-e2t' already exists and is not an empty directory.


In [4]:
%cd gated-e2t/pretraining

/workspace/gated-e2t/pretraining


In [5]:
!pip install numpy

In [6]:
pip install pandas scipy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [8]:
!pip install tqdm tensorboard matplotlib seaborn pyyaml

In [14]:
!pip install timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 20.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 6.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 44.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [timm]6/7 [timm]ngface_hub]


In [3]:
!python run_pretrain.py \
  --data_path ../data/tmp/zuco_eeg_128ch_1280len.df \
  --epochs 300 \
  --batch_size 32 \
  --output_dir Results/GLIM_Pretrain1

2026-02-23 18:06:32,249 | INFO : Using device: cuda:0
2026-02-23 18:06:32,250 | INFO : Loading data from ../data/tmp/zuco_eeg_128ch_1280len.df (memory-mapped)
2026-02-23 18:06:32,250 | INFO : Loading train split from ../data/tmp/zuco_eeg_128ch_1280len.df (memory-mapped)
^C
Traceback (most recent call last):
  File "/workspace/gated-e2t/pretraining/run_pretrain.py", line 246, in <module>
    main()
  File "/workspace/gated-e2t/pretraining/run_pretrain.py", line 155, in main
    datasets = load_zuco_memmap(args.data_path, seed=args.seed)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspace/gated-e2t/pretraining/Dataset/zuco_memmap.py", line 93, in load_zuco_memmap
    'train': ZuCoMemMapDataset(data_path, 'train', val_ratio, test_ratio, seed),
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspace/gated-e2t/pretraining/Dataset/zuco_memmap.py", line 36, in __init__
    df = pd.read_pickle(data_path)
         ^^^^^^^

In [8]:
!python run_pretrain.py \
  --n_blocks 8 \
  --emb_size 256 \
  --num_heads 16 \
  --mask_ratio 0.70 \
  --epochs 200 \
  --batch_size 64 \
  --lr 1e-4 \
  --weight_decay 0.05 \
  --linear_probe_interval 10 \
  --output_dir Results/GLIM_Pretrain_v5.2 \
  --gpu 0

2026-02-23 18:14:28,354 | INFO : Using device: cuda:0
2026-02-23 18:14:28,355 | INFO : Loading data from ../data/tmp/zuco_eeg_128ch_1280len.df (memory-mapped)
2026-02-23 18:14:28,355 | INFO : Loading train split from ../data/tmp/zuco_eeg_128ch_1280len.df (memory-mapped)
2026-02-23 18:14:55,708 | INFO : Train samples: 17869
2026-02-23 18:14:55,708 | INFO : Loading val split from ../data/tmp/zuco_eeg_128ch_1280len.df (memory-mapped)
2026-02-23 18:15:00,290 | INFO : Val samples: 2233
2026-02-23 18:15:00,290 | INFO : Loading test split from ../data/tmp/zuco_eeg_128ch_1280len.df (memory-mapped)
2026-02-23 18:15:05,018 | INFO : Test samples: 2233
2026-02-23 18:15:05,018 | INFO : Train samples: 17869, Test samples: 2233
2026-02-23 18:15:05,269 | INFO : Model parameters: 8,161,280
2026-02-23 18:15:05,279 | INFO : Starting pretraining...
Epoch 1: 100%|█████| 280/280 [00:11<00:00, 24.75it/s, loss=2.9217, align=2.0482]
2026-02-23 18:15:16,593 | INFO : Epoch 1: loss=2.5145, align=1.6379, std=0.857

In [12]:
!python evaluate_multitask_probe.py \
  --ckpt Results/GLIM_Pretrain_v5.2/best_model.pth \
  --data ../data/tmp/zuco_eeg_label_8variants.df \
  --gpu 0

  MULTI-TASK LINEAR PROBE EVALUATION
  Pretrained JEPA Encoder

📦 Loading checkpoint: Results/GLIM_Pretrain_v5.2/best_model.pth
   Epoch: 120
   Config: emb_size=256, n_blocks=8, patch_size=8, num_heads=16
   ✅ Model loaded (8,161,280 parameters)
📂 Loading data from ../data/tmp/zuco_eeg_label_8variants.df
   Total samples: 22335
   Columns: ['eeg', 'mask', 'subject', 'label id', 'raw text', 'dataset', 'task', 'control', 'raw label', 'input text', 'text uid', 'sentiment label', 'relation label', 'lexical simplification (v0)', 'lexical simplification (v1)', 'semantic clarity (v0)', 'semantic clarity (v1)', 'syntax simplification (v0)', 'syntax simplification (v1)', 'naive rewritten', 'naive simplified', 'phase']
   Train: 17908 samples | Test: 2227 samples

🔍 Extracting features from frozen encoder...
   Feature shape: (17908, 256)

📈 Feature Statistics:
   Dim: 256
   Mean std: 0.9101 | Min std: 0.7343
   ✅ No feature collapse

───────────────────────────────────────────────────────────

In [ ]:
# ls -lh workspace/gated-e2t/runs/dev-dist/wandb/latest-run/files-checkpoints/epoch=199-step=397600.ckpt